## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

## Import Datasets

In [2]:
#Load Datasets
fake_df = pd.read_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/fake_2026_06.parquet") # Update File name as needed
genuine_df = pd.read_parquet("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/genuine_2026_06.parquet") # Update File name as needed

## Fake IMEI - Analysis

In [3]:
# ==========================================
# SHARED PREP — run once for fake_df
# ==========================================
def prep_time_and_age(fake_df, time_col='last_seen'):
    fake_df['year_month'] = fake_df[time_col].dt.to_period('M')
    fake_df['hour']       = fake_df[time_col].dt.hour
    fake_df['dow']        = fake_df[time_col].dt.day_name()

    bins   = [0, 17, 30, 40, 50, 60, 70, 100, 120]
    labels = ["<18", "18-30", "31-40", "41-50", "51-60", "61-70", "71-100", "100+"]
    numeric_age = pd.to_numeric(fake_df['age'], errors='coerce')
    fake_df['age_bracket'] = pd.cut(numeric_age, bins=bins, labels=labels, right=True)
    fake_df['age_bracket'] = pd.Categorical(fake_df['age_bracket'], categories=labels, ordered=True)
    return fake_df

fake_df = prep_time_and_age(fake_df)

In [4]:
mno_col    = fake_df['mno'].astype('object').fillna('Unknown MNO')
gender_col = fake_df['gender'].fillna('Unknown Gender')

reports = {
    'Total fake per month':
        fake_df['year_month'].value_counts().sort_index().to_frame('Total Fake IMEIs'),
    'Fake per MNO by month':
        pd.crosstab(fake_df['year_month'], mno_col, margins=True, margins_name='TOTAL'),
    'Fake per gender by month':
        pd.crosstab(fake_df['year_month'], gender_col, margins=True, margins_name='TOTAL'),
    'Fake per age group by month':
        pd.crosstab(fake_df['year_month'], fake_df['age_bracket'], dropna=False,
                    margins=True, margins_name='TOTAL'),
    'Fake per district by month':
        pd.crosstab(fake_df['year_month'], fake_df['district'],
                    margins=True, margins_name='TOTAL'),
}
for title, tbl in reports.items():
    print(f"\n{title}\n{'='*70}")
    display(tbl)


Total fake per month


,Total Fake IMEIs
year_month,
1970-01,148872
2025-09,44200
2025-10,124756
2025-11,157434
2025-12,324906
2026-01,254839
2026-02,1706562
2026-03,511300
2026-05,2174025



Fake per MNO by month


mno,AIRTEL,MTN,Unknown MNO,TOTAL
year_month,,,,
1970-01,0,0,148872,148872
2025-09,17194,27006,0,44200
2025-10,70787,53969,0,124756
2025-11,89204,68230,0,157434
2025-12,162406,162500,0,324906
2026-01,149808,105031,0,254839
2026-02,942427,764135,0,1706562
2026-03,503171,8129,0,511300
2026-05,311850,1862175,0,2174025



Fake per gender by month


gender,Female,Male,Undefined,Unknown Gender,TOTAL
year_month,,,,,
1970-01,0,0,0,148872,148872
2025-09,17282,16868,0,10050,44200
2025-10,44307,41855,1,38593,124756
2025-11,54242,52008,0,51184,157434
2025-12,111509,114597,3,98797,324906
2026-01,88813,82569,0,83457,254839
2026-02,565986,562230,11,578335,1706562
2026-03,102267,94334,3,314696,511300
2026-05,872608,931371,26,370020,2174025



Fake per age group by month


age_bracket,<18,18-30,31-40,41-50,51-60,61-70,71-100,100+,TOTAL
year_month,,,,,,,,,
1970-01,0,0,0,0,0,0,0,0,148872
2025-09,1,11247,10807,6591,3388,1433,682,0,44200
2025-10,3,27710,26652,15937,10346,3683,1828,0,124756
2025-11,7,34940,33601,19800,11048,4611,2240,0,157434
2025-12,17,77607,70678,42483,22572,8950,3798,0,324906
2026-01,17,56474,52673,32907,18232,7653,3418,0,254839
2026-02,68,379233,353901,214005,115615,45765,19609,0,1706562
2026-03,12,65476,60296,37561,21150,8485,3623,0,511300
2026-05,96,614940,576291,338627,179002,67551,27430,0,2174025



Fake per district by month


district,ABIM,ADJUMANI,AGAGO,ALEBTONG,AMOLATAR,AMUDAT,AMURIA,AMURU,APAC,ARUA,...,SHEEMA,SIRONKO,SOROTI,SSEMBABULE,TORORO,UNKNOWN,WAKISO,YUMBE,ZOMBO,TOTAL
year_month,,,,,,,,,,,,,,,,,,,,,
2025-09,63,125,138,248,109,27,193,112,255,645,...,322,336,268,255,607,129,1481,217,235,34150
2025-10,128,241,348,556,250,48,402,302,505,1522,...,709,886,456,635,1420,287,5640,543,611,86163
2025-11,161,383,364,722,263,57,561,356,706,1870,...,931,1022,652,829,1790,348,4897,657,853,106250
2025-12,416,701,934,1855,821,84,1592,944,1818,4348,...,2135,2489,1701,1514,3457,716,8112,1406,1681,226109
2026-01,293,645,676,1163,465,57,840,539,1260,3129,...,1591,1773,1048,1307,2790,580,7859,1318,1308,171382
2026-02,1694,2693,4458,7773,3526,243,5000,3295,7702,17940,...,12936,11405,5827,8629,18788,3521,53857,7424,7326,1128227
2026-03,117,169,237,391,240,31,402,397,429,1228,...,1993,2248,632,1884,3525,672,11367,353,665,196604
2026-05,3017,4982,7799,14533,6917,344,9618,5748,14226,32160,...,22685,17753,12022,12804,31099,5369,79038,13154,12467,1804005
TOTAL,5889,9939,14954,27241,12591,891,18608,11693,26901,62842,...,43302,37912,22606,27857,63476,11622,172251,25072,25146,3752890


In [5]:
# 1. IMEI REUSE / CLONING — one fake IMEI seen across many MSISDNs is a strong cloning signal
imei_spread = (fake_df.groupby('imei')['msisdn'].nunique()
               .sort_values(ascending=False).rename('distinct_msisdns'))
print("Top suspected cloned IMEIs (1 IMEI, many numbers):")
display(imei_spread.head(20).to_frame())

# 2. IDENTITY-FARMING — one id_number tied to many MSISDNs on fake devices
#    (you saw this exact pattern earlier: NUBUWATI LWANGA on 4 sequential numbers)
id_spread = (fake_df[fake_df['id_type'] == 'NATIONAL_ID']
             .groupby('id_number')['msisdn'].nunique()
             .sort_values(ascending=False).rename('numbers_per_id'))
print("Identities linked to the most fake-device numbers:")
display(id_spread[id_spread > 1].head(20).to_frame())

# 3. STATUS MIX over time — is the blacklist (B) growing vs greylist/white (W)?
status_trend = pd.crosstab(fake_df['year_month'], fake_df['imei_status'], normalize='index')
display(status_trend.style.format('{:.1%}'))

# 4. AGE-PLAUSIBILITY flag — impossible/implausible ages signal bad/fraudulent registration
implausible = fake_df[(pd.to_numeric(fake_df['age'], errors='coerce') < 10) |
                      (pd.to_numeric(fake_df['age'], errors='coerce') > 100)]
print(f"Registrations with implausible age: {len(implausible):,}")

# 5. TIME-OF-DAY fingerprint — bulk fraudulent activations often cluster at odd hours
hourly = fake_df['hour'].value_counts().sort_index()
display(hourly.to_frame('detections'))

# 6. GEOGRAPHIC concentration — share of fakes by district (where to focus enforcement)
district_share = (fake_df['district'].value_counts(normalize=True)
                  .head(15).mul(100).round(2).rename('% of fakes'))
display(district_share.to_frame())

Top suspected cloned IMEIs (1 IMEI, many numbers):


,distinct_msisdns
imei,
35778967740070,8
00000000000000,7
00440015202000,7
35045678912345,7
35098765432456,7
35123638438836,7
35181898765432,7
35198765432123,7
35211865432876,7


Identities linked to the most fake-device numbers:


,numbers_per_id
id_number,
CM991091049YHL,9
CM65027100CHXL,8
CF520521081DXE,8
CM78103102AXCC,8
CF95015102R4LK,8
CM77031107R54D,8
CM920331058URK,8
CM88023106848H,8
CF88032106A7UD,7


imei_status,,B,W
year_month,,,
1970-01,100.0%,0.0%,0.0%
2025-09,0.0%,0.0%,100.0%
2025-10,0.0%,0.1%,99.9%
2025-11,0.0%,0.0%,100.0%
2025-12,0.0%,0.0%,100.0%
2026-01,0.0%,0.1%,99.9%
2026-02,0.0%,0.0%,100.0%
2026-03,0.0%,0.0%,100.0%
2026-05,0.0%,0.0%,100.0%


Registrations with implausible age: 225


,detections
hour,
0,213012
1,45911
2,31068
3,28147
4,31850
5,48595
6,82108
7,119907
8,135729


,% of fakes
district,
WAKISO,4.59
LUWEERO,2.82
KASESE,2.72
MUKONO,2.58
IGANGA,1.96
KABAROLE,1.96
KABALE,1.95
NTUNGAMO,1.95
MITYANA,1.91


In [8]:
# SHARED-DEVICE RINGS: clusters where multiple IMEIs + multiple IDs + multiple numbers
# interlock. A genuine person has ~1 ID, few devices. A SIM-box/registration ring shows
# many-to-many. Build an edge list and measure component size.
ring = (fake_df.groupby('id_number')
        .agg(n_msisdn=('msisdn','nunique'),
             n_imei=('imei','nunique'),
             n_districts=('district','nunique'))
        .query('n_msisdn > 3 and n_imei > 1')
        .sort_values('n_msisdn', ascending=False))
display(ring.head(25))

# SEQUENTIAL-NUMBER FARMING: same ID across consecutive MSISDNs (you saw NUBUWATI's 4-in-a-row).
# Bulk-registered blocks are a tell.
s = fake_df.sort_values('msisdn').copy()
s['msisdn_int'] = pd.to_numeric(s['msisdn'], errors='coerce')
s['gap'] = s.groupby('id_number')['msisdn_int'].diff()
sequential = s[s['gap'] == 1]['id_number'].value_counts()
print("IDs holding runs of consecutive numbers:")
display(sequential[sequential > 0].head(20).to_frame('consecutive_links'))

,n_msisdn,n_imei,n_districts
id_number,,,
CERT_UTEL,24584,24599,0
CERT_215551,243,243,0
CERT_80020002196944,237,231,0
1,126,129,0
CERT_80020002390234,70,64,0
CCIT354,60,62,0
CERT_80010000386201,48,50,0
80020003435673,46,47,0
CERT_80020001145672,41,44,0


IDs holding runs of consecutive numbers:


,consecutive_links
id_number,
CERT_UTEL,14257
CERT_215551,8
80020003435673,2
CM81032102WYME,2
CERT_80020001145672,2
CM89108102M6CE,1
CM000221075WNH,1
CCIT354,1
CM85061102HJCJ,1


In [11]:
# ACTIVATION VELOCITY SPIKES: detect months where a district's fake count jumps far above
# its own trailing average — early-warning for a fraud campaign.
monthly = (fake_df.groupby(['district','year_month']).size()
           .rename('n').reset_index())
monthly['roll'] = (monthly.groupby('district')['n']
                   .transform(lambda x: x.shift().rolling(3, min_periods=1).mean()))
monthly['spike_ratio'] = monthly['n'] / monthly['roll']
spikes = monthly[monthly['spike_ratio'] > 3].sort_values('spike_ratio', ascending=False)
display(spikes.head(20))

# DORMANT-THEN-ACTIVE: device last_seen patterns — long gaps then sudden reappearance can
# indicate resold/recycled stolen handsets (richer if you have first_seen too).

,district,year_month,n,roll,spike_ratio
681,MADI-OKOLLO,2025-10,20,2.000000,10.000000
1004,RWAMPARA,2026-02,334,37.666667,8.867257
1020,SHEEMA,2026-02,12936,1552.333333,8.333262
181,BUNYANGABU,2026-02,806,100.000000,8.060000
213,BUTAMBALA,2026-02,11232,1400.000000,8.022857
437,KARENGA,2026-02,37,4.666667,7.928571
253,GOMBA,2026-02,10291,1317.666667,7.810018
781,MPIGI,2026-02,17520,2252.000000,7.779751
741,MBARARA,2026-02,20998,2700.333333,7.776077
189,BUSHENYI,2026-02,12795,1645.666667,7.774965


## Genuine IMEI - Analysis

In [6]:
# DEVICE LANDSCAPE (light — these aggregate to small tables)
brand_share   = genuine_df['brand'].value_counts(normalize=True).head(15)
os_share      = genuine_df['os_family'].value_counts(normalize=True)
device_types  = genuine_df['device_type'].value_counts()

# NETWORK-CAPABILITY / DIGITAL-DIVIDE picture — what tech is the base actually on?
capability = pd.DataFrame({
    '2G_capable': genuine_df['has_2g'].sum(),
    '3G_capable': genuine_df['has_3g'].sum(),
    '4G_capable': genuine_df['has_4g'].sum(),
    '5G_capable': genuine_df['has_5g'].sum(),
}, index=['count']).T
display(capability)

# 4G-CAPABLE BUT... cross with MNO to see which operator's base is upgrade-ready
g4_by_mno = genuine_df.groupby('mno', observed=True)['has_4g'].mean().mul(100).round(1)
display(g4_by_mno.rename('% 4G-capable').to_frame())

# DEVICE AGE / REFRESH — how old is the installed base? (years since release)
yr = pd.to_numeric(genuine_df['year_released'], errors='coerce')
genuine_df['device_age'] = 2026 - yr.where(yr > 0)
display(genuine_df['device_age'].describe().to_frame())

# DEMOGRAPHIC × DEVICE — do women/men, or age groups, skew to different brands?
genuine_df = prep_time_and_age(genuine_df)  # adds age_bracket
brand_by_age = pd.crosstab(genuine_df['age_bracket'],
                           genuine_df['brand'].where(
                               genuine_df['brand'].isin(brand_share.index)),
                           normalize='index', dropna=False)
display(brand_by_age.style.format('{:.1%}'))

,count
2G_capable,105142103
3G_capable,51473117
4G_capable,44903639
5G_capable,3758352


,% 4G-capable
mno,
AIRTEL,30.2
HAMILTON,88.6
MTN,44.2


,device_age
count,98504708.0
mean,5.627751
std,2.782475
min,1.0
25%,3.0
50%,6.0
75%,8.0
max,32.0


brand,Apple,Infinix,Mione,NOKIA,OPPO,Redmi,SIMCom,SIMIX,Samsung,TECNO,VILLAON,ZTE,itel,kgtel,vivo,
age_bracket,,,,,,,,,,,,,,,,
nan,4.1%,6.6%,1.0%,2.2%,2.1%,2.4%,5.2%,1.4%,20.3%,24.5%,1.4%,1.3%,25.5%,0.6%,1.4%,0.0%
<18,0.7%,2.3%,1.6%,2.6%,0.7%,0.9%,0.0%,1.9%,8.0%,24.7%,2.8%,0.2%,52.3%,1.0%,0.2%,0.0%
18-30,2.0%,5.4%,1.4%,2.2%,1.2%,1.3%,0.0%,1.6%,12.6%,24.6%,2.6%,0.7%,42.8%,0.8%,0.8%,0.0%
31-40,1.5%,5.0%,1.4%,2.3%,0.9%,1.2%,0.0%,1.5%,11.9%,25.9%,2.4%,0.6%,43.8%,0.8%,0.6%,0.0%
41-50,1.0%,4.3%,1.4%,2.1%,0.9%,1.1%,0.0%,1.5%,10.3%,26.6%,2.4%,0.6%,46.4%,0.8%,0.6%,0.0%
51-60,0.8%,3.8%,1.4%,2.1%,0.8%,1.0%,0.0%,1.5%,8.8%,26.8%,2.5%,0.5%,48.7%,0.9%,0.5%,0.0%
61-70,0.6%,3.1%,1.4%,2.3%,0.7%,0.9%,0.0%,1.5%,7.6%,26.8%,2.5%,0.4%,50.7%,0.9%,0.5%,0.0%
71-100,0.4%,2.4%,1.4%,2.7%,0.6%,0.7%,0.0%,1.6%,6.6%,26.0%,2.6%,0.4%,53.1%,0.9%,0.4%,0.0%
100+,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%


In [ ]:
# DISTRICT GENDER SKEW: a district that is 95% male on fake devices is suspicious
# (mass-registration under harvested IDs tends to skew). Compare fake-side gender mix
# to that district's genuine-side mix.
fake_g  = pd.crosstab(fake_df['district'], fake_df['gender'], normalize='index')
gen_g   = pd.crosstab(genuine_df['district'], genuine_df['gender'], normalize='index')
gender_skew = (fake_g - gen_g).abs().sum(axis=1).sort_values(ascending=False)
print("Districts where fake-device gender mix deviates most from genuine:")
display(gender_skew.head(15).to_frame('total_abs_deviation'))

# AGE-PYRAMID DISTORTION: same idea on age brackets — KS-style divergence per district.
fake_a = pd.crosstab(fake_df['district'], fake_df['age_bracket'], normalize='index', dropna=False)
gen_a  = pd.crosstab(genuine_df['district'], genuine_df['age_bracket'], normalize='index', dropna=False)
age_skew = (fake_a - gen_a).abs().sum(axis=1).sort_values(ascending=False)
display(age_skew.head(15).to_frame('age_distribution_deviation'))

In [ ]:
# CLONE-MAGNET BRANDS: which brands are over-represented in fakes vs their market share?
# Needs brand attached to fake_df — join via imei TAC (first 8 digits) to genuine_df's device lookup.
fake_df['tac'] = fake_df['imei'].str[:8]
tac_lookup = (genuine_df[['tac','brand','model','device_type']]
              .drop_duplicates('tac'))
fake_dev = fake_df.merge(tac_lookup, on='tac', how='left')

fake_brand  = fake_dev['brand'].value_counts(normalize=True)
gen_brand   = genuine_df['brand'].value_counts(normalize=True)
lift = (fake_brand / gen_brand).dropna().sort_values(ascending=False)
print("Brand fraud-lift (>1 = over-represented among fakes):")
display(lift.head(20).to_frame('fraud_lift'))

# 'NOT KNOWN' / unrecognised TACs among fakes — a fabricated IMEI often has no real TAC.
unmatched_tac = fake_dev['brand'].isna().mean()
print(f"Share of fake IMEIs with no matching real device profile: {unmatched_tac:.1%}")

In [12]:
# UPGRADE-READY BASE: 4G/5G-capable devices still riding 2G/3G — the operator's
# realistic upgrade target. Cross capability with district to map where to push 4G.
upgrade_target = genuine_df[(genuine_df['has_4g']==1)]  # capable
target_by_district = (upgrade_target.groupby('district', observed=True).size()
                      .sort_values(ascending=False))
display(target_by_district.head(20).to_frame('4G_capable_devices'))

# DEVICE-AGE vs DEMOGRAPHICS: are older citizens on older hardware? (digital-divide story)
genuine_df['device_age'] = 2026 - pd.to_numeric(genuine_df['year_released'],
                                                errors='coerce').where(lambda x: x>0)
age_vs_device = genuine_df.groupby('age_bracket', observed=True)['device_age'].median()
display(age_vs_device.to_frame('median_device_age_yrs'))

,4G_capable_devices
district,
WAKISO,1237683
KASESE,786730
LUWEERO,721400
KABALE,658026
MUKONO,649426
NTUNGAMO,617718
ARUA,577272
KABAROLE,561443
MBARARA,518318


,median_device_age_yrs
age_bracket,
<18,7.0
18-30,5.0
31-40,6.0
41-50,7.0
51-60,7.0
61-70,7.0
71-100,7.0


In [13]:
# MULTI-SIM PENETRATION: distinct MSISDNs per id_number in the genuine base — the legitimate
# baseline against which fake_df's farming looks abnormal. (Heavy — do in ClickHouse.)
# SELECT id_number, count(distinct msisdn) sims FROM genuine GROUP BY id_number

# COVERAGE GAP: numbers in fake_df whose id_number NEVER appears in genuine_df —
# i.e. identities that only ever show up attached to fake devices. Strong fraud signal.
genuine_ids = set(genuine_df['id_number'].dropna().unique())   # large set; consider CH
only_fake = fake_df[~fake_df['id_number'].isin(genuine_ids)]
print(f"Identities seen ONLY on fake devices: {only_fake['id_number'].nunique():,}")

Identities seen ONLY on fake devices: 125,567
